# Strobe imaging of membrane charging (Figure 4), a check on DANDI:001273

Paper: Semenov et al. (2025), Cell Reports Physical Science 6, 102660, doi:10.1016/j.xcrp.2025.102660.
Dataset: DANDI:001273, "Kinetics of Membrane Charging and Its Dependence on Neuron Shape and Alignment
with the Electric Field", version 0.250805.1356, the most recent published version (the paper cites
none).

The claims fig4-pooled-time-constants and fig4-orientation-and-poles of
`claims/001272/10.1016-j.xcrp.2025.102660.yaml` defer their fits and readings to the claims file for
DANDI:001273. This notebook checks whether those fits can be made from the deposited files: it lists
the experiments, and runs the fits of that claims file's header on one experiment. Both claims are
blocked in the attempt record; this is the evidence.

| Claim | Here |
|---|---|
| fig4-pooled-time-constants | blocked (metadata insufficient), evidence below |
| fig4-orientation-and-poles | blocked (metadata insufficient), evidence below |

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import strobe as sb

assets = sb.list_assets()
with ThreadPoolExecutor(16) as ex:
    M = list(ex.map(sb.read_meta, assets))
meta = pd.DataFrame(M)
print("dandiset", sb.DANDISET, "version", sb.VERSION, "|", len(meta), "files")

In [ ]:
print("frames per file:", meta["shape"].astype(str).value_counts().to_dict())
print("cells (subjects):", meta.subject.nunique(), "| pulse length (us):", sorted(meta.pulse_us.unique()),
      "| nominal volts:", sorted(meta.volts.unique()))
groups = meta.groupby(["subject", "electrodes", "volts"]).agg(files=("path", "size"), backward=("backward", "sum"))
print(groups.to_string())
full = groups[groups.files >= 7]
print(f"groups of 7 or 8 replicates (experiments): {len(full)}; at 40 V nominal: {int((full.index.get_level_values('volts') == 40).sum())}; "
      f"single files: {int((groups.files == 1).sum())}; the paper reports 19 experiments")
print("electrode orientation labels:", meta.electrodes.value_counts().to_dict(),
      "| cell shape labels (elongated or random): none in the files")

## One experiment

Cell 1_20240119 with the electrodes right and left of it, 8 replicates at 40 V nominal. The
claims file's ROIs are rebuilt from the RATIO image (001273-O1 a and b), the traces are averaged over
the replicates (001273-F3) and fitted under the weighting and amplitude readings (001273-O2, O3). The
paper scaled the ROI data to a "debris" ROI to remove laser-flash fluctuations (001273-F2); no debris
ROI is deposited, so the traces are unscaled, and, as a variant made here, scaled by the whole-frame
mean of each frame.

In [ ]:
sel = meta[(meta.subject == "1_20240119") & (meta.electrodes == "right_left") & (meta.volts == 40)]
E = sb.load_experiment(sel.to_dict("records"))
t = E["t"]
print("replicates:", E["n_replicates"], "| cell pixels:", E["cell_px"], "| ROI pixels:",
      {f"{p}/{r}": v for (p, r), v in E["roi_px"].items()})
print("RATIO extremes in the cell:", {p: round(E["rois"][(p, "extreme")], 3) for p in ("cathodic", "anodic")})
fm = E["frame_means"]
f0 = np.nanmean(fm[:, t < 0], axis=1, keepdims=True)
whole = 100 * (fm - f0) / f0
print(f"whole-frame mean, replicate mean: SD over frames {np.nanstd(np.nanmean(whole, 0)):.2f} %, "
      f"range {np.nanmin(np.nanmean(whole, 0)):.1f} to {np.nanmax(np.nanmean(whole, 0)):.1f} %")

In [ ]:
rows = []
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
for j, scaling in enumerate(("unscaled", "whole-frame scaled (variant)")):
    for key, v in E["traces"].items():
        s = v if j == 0 else 100 * ((1 + v / 100) / (1 + whole / 100) - 1)
        y = np.nanmean(s, axis=0)
        pre_sd = np.nanstd(y[t < 0])
        late = np.nanmean(y[(t >= 1.5) & (t < 2.0)])
        ax = axes[j, 0 if key[0] == "cathodic" else 1]
        ax.plot(t, y, lw=0.8, label=f"ROI {key[1]}")
        ax.axvspan(0, 2, color="tab:blue", alpha=0.08)
        ax.set_title(f"{key[0]}, {scaling}", fontsize=9)
        for w in "ab":
            for d in "ab":
                f = sb.fit_tau(t, s, w, d)
                rows.append(dict(scaling=scaling, pole=key[0], roi=key[1], weighting=w, discharge_amp=d,
                                 pre_sd=round(pre_sd, 2), late_pulse_dF=round(late, 2),
                                 tau_charging=round(f["tau_c"], 3), tau_discharging=round(f["tau_d"], 3),
                                 amp=round(f["amp_c"], 2)))
for ax in axes.ravel():
    ax.set_ylim(-15, 15); ax.legend(fontsize=7)
axes[1, 0].set_xlabel("time from pulse onset (us)"); axes[1, 1].set_xlabel("time from pulse onset (us)")
plt.tight_layout(); plt.show()
fits = pd.DataFrame(rows)
print(fits.to_string(index=False))
for sc, g in fits.groupby("scaling"):
    print(f"{sc}: charging tau {g.tau_charging.min()} to {g.tau_charging.max()} us, "
          f"discharging tau {g.tau_discharging.min()} to {g.tau_discharging.max()} us over the readings; "
          f"noise before the pulse (SD) {g.pre_sd.min():.2f} to {g.pre_sd.max():.2f} %, dF late in the pulse "
          f"{g.late_pulse_dF.min():.2f} to {g.late_pulse_dF.max():.2f} %")

**Result: blocked.** In this experiment the replicate-mean traces are dominated by frame-to-frame
laser-flash fluctuation common to both poles, which the paper removed by scaling to a debris ROI that
is not deposited. The noise before the pulse (SD 2.8 to 3.0% unscaled, 1.2 to 1.5% scaled
by the whole frame) is of the order of the polarization late in the pulse (2.6 to 5.5% in magnitude),
and the fitted time constants range from 0.26 to 20 us for charging and from 0.02 to 20 us for
discharging across the claims file's readings (table above). The readings cannot agree, and the fits
do not measure the sub-microsecond charging the claim is about. The files also hold 22 experiments of
7 or 8 replicates, one pair and 4 single files, where the paper reports 19 experiments, with no
mapping, and no cell-shape labels for the orientation groups.

## Summary

| Claim | Outcome |
|---|---|
| fig4-pooled-time-constants | blocked: metadata insufficient |
| fig4-orientation-and-poles | blocked: metadata insufficient |

What was found about the dataset: the debris ROI the paper scales to is not deposited; the
experiments do not map to the paper's 19; the cell shapes (elongated or random) are not labelled; the
first acquired frame of every file is dark.